# Colab AI Model Stüdyosu — DeepSeek V4.1 Flash / GLM-5.3 / Yerel Abliterated

**API ve yerel GPU desteği olan Colab defteri.**

- **DeepSeek V4.1 Flash**: Resmî DeepSeek API'si, `deepseek-flash`.
- **GLM-5.3 / GLM-5.3 Flash**: OpenRouter üzerinden standart model sürümleri.
- **Qwen3-8B Abliterated Q4_K_M**: Colab GPU üzerinde gerçek yerel çıkarım; yaklaşık 5 GB model ağırlığı.
- **Özel sağlayıcı**: Uncensored GLM-5.3 gibi büyük modelleri gerçekten sunan **kendi OpenAI-uyumlu API sağlayıcınız** varsa kullanın.

**Önemli:** GLM-5.3 Uncensored (753B) ve DeepSeek V4.1 Flash (552B omurga) tek Colab Pro GPU belleğine sığmaz. API modları uzakta çalışır ve ayrıca ücretlidir. Standart GLM-5.3 API modeli Uncensored sürüm değildir.

**Kurulum**: 1. hücreyi, sonra 2. hücreyi çalıştırın. GPU'da yerel Qwen kullanacaksanız aradaki **Yerel model (opsiyonel)** hücresini de çalıştırın. API kullanacaksanız Colab sol menüsünde **Secrets / Gizli anahtarlar** altında `DEEPSEEK_API_KEY`, `OPENROUTER_API_KEY` veya `CUSTOM_API_KEY` tanımlayın ve erişimi etkinleştirin. API-only çalışmada CPU yeterlidir. Son hücre Gradio'ya parola korumalı, geçici bir URL üretir; Colab oturumu kapanınca URL çalışmaz.

Kaynaklar: [DeepSeek API](https://api-docs.deepseek.com/guides/codex) · [OpenRouter GLM-5.3](https://openrouter.ai/z-ai/glm-5.3) · [GLM-5.3 Uncensored](https://huggingface.co/dealignai/GLM-5.3-UNCENSORED-FP8) · [llama.cpp](https://github.com/ggml-org/llama.cpp).

## 1 — Bağımlılıkları kur

In [ ]:
%pip -q install --upgrade openai gradio huggingface_hub

## Yerel model — opsiyonel (yalnız GPU modunda)

Büyük GLM/DeepSeek modelleri bu hücrede indirilmez. Sadece **Qwen3-8B Abliterated** 5.03 GB GGUF ağırlığı Colab GPU'suna yüklenir. CUDA kurulu GPU çalışma zamanı gereklidir. Sadece API ile çalışacaksanız bu hücreyi atlayın.

In [ ]:
import shutil, subprocess, pathlib, urllib.request, time
from huggingface_hub import hf_hub_download
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Çalışma zamanı > Çalışma zamanı türünü değiştir > GPU seçin.")
subprocess.run(["bash","-lc","apt-get update -qq && apt-get install -qq -y cmake ninja-build build-essential libcurl4-openssl-dev"],check=True)
src=pathlib.Path("/content/llama.cpp")
if not src.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/ggml-org/llama.cpp.git",str(src)],check=True)
subprocess.run(["cmake","-S",str(src),"-B",str(src/"build"),"-DGGML_CUDA=ON","-DCMAKE_BUILD_TYPE=Release","-DLLAMA_CURL=ON"],check=True)
subprocess.run(["cmake","--build",str(src/"build"),"--target","llama-server","-j","2"],check=True)
model_path=hf_hub_download(repo_id="richardyoung/Qwen3-8B-Abliterated-GGUF",filename="qwen3-8b-abliterated-Q4_K_M.gguf",local_dir="/content/models/qwen")
try:
    urllib.request.urlopen("http://127.0.0.1:8001/v1/models",timeout=2).close()
    print("Sunucu zaten çalışıyor.")
except Exception:
    log=open("/content/llama-server.log","a",buffering=1)
    process=subprocess.Popen([str(src/"build/bin/llama-server"),"-m",model_path,"--host","127.0.0.1","--port","8001","--alias","qwen3-8b-abliterated","--ctx-size","8192","--n-gpu-layers","99"],stdout=log,stderr=subprocess.STDOUT)
    for _ in range(120):
        if process.poll() is not None: break
        try:
            urllib.request.urlopen("http://127.0.0.1:8001/v1/models",timeout=2).close()
            print("Yerel Qwen3-8B GPU sunucusu çalışıyor.")
            break
        except Exception: time.sleep(2)
    else:
        raise RuntimeError("Sunucu başlamadı: /content/llama-server.log dosyasını inceleyin.")

## 2 — Parola korumalı sohbet arayüzünü başlat

API anahtarları Colab **Secrets** içinden okunur, notebook'a kaydedilmez. Sonuçta yazılan Gradio adresi yalnızca mevcut oturum boyunca geçerlidir. Oturum sonlandırıldığında bağlantı kapanır.

In [ ]:
import os, secrets, urllib.request
import gradio as gr
from openai import OpenAI
try:
    from google.colab import userdata
except ImportError:
    userdata=None

def getkey(name):
    if name is None: return "local"
    k=os.environ.get(name,"")
    if not k and userdata is not None:
        try: k=userdata.get(name) or ""
        except Exception: pass
    return k

models={
    "DeepSeek V4.1 Flash (resmî API)":("https://api.deepseek.com","deepseek-flash","DEEPSEEK_API_KEY"),
    "GLM-5.3 (standart/OpenRouter)":("https://openrouter.ai/api/v1","z-ai/glm-5.3","OPENROUTER_API_KEY"),
    "GLM-5.3 Flash (standart/OpenRouter)":("https://openrouter.ai/api/v1","z-ai/glm-5.3-flash","OPENROUTER_API_KEY"),
    "DeepSeek V4.1 Flash (OpenRouter)":("https://openrouter.ai/api/v1","deepseek/deepseek-v4.1-flash","OPENROUTER_API_KEY"),
    "Yerel Qwen3-8B Abliterated":("http://127.0.0.1:8001/v1","qwen3-8b-abliterated",None),
    "Özel OpenAI-uyumlu sağlayıcı (ör. GLM Uncensored)":(None,None,"CUSTOM_API_KEY")
}
def answer(message,history,selection,custom_url,custom_model):
    base,model,secret=models[selection]
    if base is None:
        base,model=custom_url.strip(),custom_model.strip()
        if not base.startswith(("https://","http://localhost:","http://127.0.0.1:")) or not model:
            return "Özel sağlayıcı için HTTPS base_url ve model kimliği belirtin."
    key=getkey(secret)
    if not key:
        return f"Sol menüde Colab Secrets bölümüne {secret} ekleyin ve deftere erişimi etkinleştirin."
    if secret is None:
        try: urllib.request.urlopen("http://127.0.0.1:8001/v1/models",timeout=2).close()
        except Exception: return "Yerel model çalışmıyor. Önce opsiyonel yerel model hücresini çalıştırın."
    messages=[{"role":"system","content":"Respond accurately in the user's language."}]
    for h in history[-20:]:
        if isinstance(h,dict) and h.get("role") in ("user","assistant") and isinstance(h.get("content"),str):
            messages.append({"role":h["role"],"content":h["content"]})
    messages.append({"role":"user","content":message})
    try:
        result=OpenAI(api_key=key,base_url=base,timeout=180,max_retries=2).chat.completions.create(model=model,messages=messages,max_tokens=4096)
        return result.choices[0].message.content or "Sağlayıcı boş yanıt döndürdü; token veya model ayarını kontrol edin."
    except Exception as e:
        return f"{type(e).__name__}: {e}"

with gr.Blocks(title="Colab AI Model Stüdyosu") as demo:
    gr.Markdown("# Colab AI Model Stüdyosu\nBüyük modeller API ile, Qwen3-8B ise kendi Colab GPU'nuzda çalışır.")
    choice=gr.Dropdown(list(models),value="DeepSeek V4.1 Flash (resmî API)",label="Model")
    with gr.Accordion("Özel API URL/model",open=False):
        url=gr.Textbox(label="OpenAI-uyumlu base_url",placeholder="https://ornek-saglayici.com/v1")
        model_id=gr.Textbox(label="Model kimliği",placeholder="sağlayici-model-id")
    gr.ChatInterface(fn=answer,type="messages",additional_inputs=[choice,url,model_id],description="Anahtarları yalnızca Colab Secrets içine girin.")
password=secrets.token_urlsafe(18)
print("Gradio kullanıcı: colab")
print("Gradio parola:",password)
print("Aşağıdaki geçici bağlantı parolayla korunur. Bağlantıyı üçüncü kişilerle paylaşmayın.")
demo.launch(share=True,inline=True,auth=("colab",password),show_error=True)

### Sınırlar
- MCP aracılığıyla Colab GPU'suna buradan otomatik oturum açılması/çalışma zamanı başlatılması desteklenmedi. GPU'yu Colab içinde sizin başlatmanız gerekir.
- API anahtarları Colab Pro aboneliğine dahil değildir.
- Uncensored GLM-5.3 modeli sadece o ağırlıkları gerçekten barındıran özel bir sağlayıcıyla kullanılabilir. Standart GLM-5.3 API'yi bu isimle pazarlamayın.
- Geçici Gradio URL'si parola korumalı olsa da internete açıktır. İşiniz bittiğinde çalışma zamanını kapatın.
- Bu defter Colab runtime'ında uçtan uca çalıştırılarak test edilmedi; hazır çalışma ortamında donanım/paket uyarlaması gerekebilir.